# EvolveGCN-H Diagnostic Investigation

Comprehensive evidence notebook for diagnosing prediction collapse and representation quality in the EvolveGCN-H CAMELS-SIMBA Ωm experiment.

All tables below are loaded from JSON diagnostics in the experiment folder. No diagnostic values are manually typed into the notebook narrative.

In [1]:
import json
import pandas as pd
from pathlib import Path

EXP_DIR = Path("experiments/evolvegcn_h_500u_top500_h32_seed42")
if not EXP_DIR.exists():
    EXP_DIR = Path("../../experiments/evolvegcn_h_500u_top500_h32_seed42")

DIAG_DIR = EXP_DIR / "diagnostics"

def load_json(filename):
    path = DIAG_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Diagnostic JSON not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def round_df(df, digits=4):
    return df.round(digits)

print("Experiment directory:", EXP_DIR)
print("Diagnostics directory:", DIAG_DIR)
print("Diagnostics directory exists:", DIAG_DIR.exists())
print("\nAvailable JSON files:")
for path in sorted(DIAG_DIR.glob("*.json")):
    print("-", path.name)

Experiment directory: ../../experiments/evolvegcn_h_500u_top500_h32_seed42
Diagnostics directory: ../../experiments/evolvegcn_h_500u_top500_h32_seed42/diagnostics
Diagnostics directory exists: True

Available JSON files:
- embedding_distribution_shift.json
- embedding_feature_stability.json
- embedding_neighborhood_consistency.json
- embedding_probe_splits.json
- graph_vs_summary_baseline.json
- head_vs_optimal_linear_solution.json
- split_target_distribution.json
- summary_vs_embedding_combined.json
- test_embedding_target_relationship.json
- test_feature_variance.json
- test_head_analysis.json
- test_layer1_activations.json
- test_layer_variance.json
- test_regressor_head_stats.json
- test_representation_stats.json
- test_variance_flow.json


# 1. Initial Problem Statement

Observed behavior:

- EvolveGCN-H predictions are less variable than the true Ωm targets.
- Simple universe-level summary features can outperform the trained GNN on the same task.
- The purpose of this notebook is to identify whether the bottleneck is the data split, embedding representation, pooling, or regression head.

# 2. Dataset and Target Verification

This diagnostic checks whether the saved train/validation/test split has non-constant Ωm targets and whether the target ranges are comparable across splits.

In [2]:
split_dist = load_json("split_target_distribution.json")

rows = []
for split, stats in split_dist.get("splits", {}).items():
    q = stats.get("quantiles", {})
    rows.append({
        "split": split,
        "n": stats.get("num_samples"),
        "mean": stats.get("mean"),
        "std": stats.get("std"),
        "min": stats.get("min"),
        "q05": q.get("0.05"),
        "median": q.get("0.50"),
        "q95": q.get("0.95"),
        "max": stats.get("max"),
    })

target_df = pd.DataFrame(rows)
display(round_df(target_df, 4))

comparison_rows = []
for pair, stats in split_dist.get("pairwise_comparisons", {}).items():
    ks = stats.get("ks_test", {})
    hist = stats.get("histogram_overlap", {})
    interval = stats.get("interval_overlap", {})
    comparison_rows.append({
        "pair": pair,
        "ks_statistic": ks.get("statistic"),
        "ks_p_value": ks.get("p_value_asymptotic"),
        "hist_overlap_%": hist.get("histogram_overlap_percent"),
        "union_range_overlap_%": interval.get("union_range_overlap_percent"),
    })

display(round_df(pd.DataFrame(comparison_rows), 4))

,split,n,mean,std,min,q05,median,q95,max
0,train,350,0.2998,0.1167,0.1018,0.1196,0.2988,0.4784,0.4998
1,val,75,0.3046,0.1122,0.1042,0.1125,0.3074,0.4814,0.4994
2,test,75,0.2917,0.1013,0.1070,0.1220,0.2914,0.4435,0.4814


,pair,ks_statistic,ks_p_value,hist_overlap_%,union_range_overlap_%
0,train_vs_val,0.0895,0.6852,87.9048,99.2965
1,train_vs_test,0.1162,0.3543,82.0000,94.0704
2,val_vs_test,0.1067,0.7638,78.6667,94.7368


Conclusion: Ωm labels are variable in every split: train std is 0.1167, validation std is 0.1122, and test std is 0.1013. The KS p-values are all above 0.35 and histogram overlap is 78.6667% to 87.9048%, so the displayed diagnostics do not support a major target-distribution mismatch as the cause of collapse.

# 3. Prediction Collapse Analysis

This diagnostic compares the prediction distribution against the true target distribution on the test split and reports representation statistics around the final prediction stage.

In [3]:
repr_stats = load_json("test_representation_stats.json")
reports = repr_stats.get("reports", {})

rows = []
for name in ["predictions", "targets"]:
    stats = reports.get(name, {})
    rows.append({
        "quantity": name,
        "mean": stats.get("mean"),
        "std": stats.get("std"),
        "min": stats.get("min"),
        "max": stats.get("max"),
        "num_values": stats.get("num_values"),
    })

pt = reports.get("prediction_target", {})
rows.append({
    "quantity": "prediction_std / target_std",
    "mean": None,
    "std": (
        pt.get("prediction_std") / pt.get("target_std")
        if pt.get("prediction_std") is not None and pt.get("target_std") not in [None, 0]
        else None
    ),
    "min": None,
    "max": None,
    "num_values": None,
})

display(round_df(pd.DataFrame(rows), 4))

stage_rows = []
for key in [
    "node_embeddings_valid_nodes",
    "graph_embeddings_after_graph_pooling",
    "universe_embeddings_after_temporal_pooling",
    "regressor_inputs",
]:
    stats = reports.get(key, {})
    stage_rows.append({
        "stage": key,
        "std": stats.get("std"),
        "avg_feature_variance": stats.get("avg_feature_variance_across_samples"),
        "pairwise_distance_per_feature": stats.get("avg_pairwise_squared_distance_per_feature"),
    })

display(round_df(pd.DataFrame(stage_rows), 6))

,quantity,mean,std,min,max,num_values
0,predictions,0.3199,0.0367,0.1329,0.3899,75.0
1,targets,0.2917,0.1013,0.1070,0.4814,75.0
2,prediction_std / target_std,NaN,0.3624,NaN,NaN,NaN


,stage,std,avg_feature_variance,pairwise_distance_per_feature
0,node_embeddings_valid_nodes,0.183781,0.009758,0.019781
1,graph_embeddings_after_graph_pooling,0.161192,0.002441,0.004948
2,universe_embeddings_after_temporal_pooling,0.065929,0.000581,0.001178
3,regressor_inputs,0.065929,0.000581,0.001178


Conclusion: prediction collapse is confirmed. On the test split, prediction std is 0.0367 while target std is 0.1013, so prediction spread is only 0.3624 of the target spread. Representation variance also drops strongly from node embeddings to graph and temporal embeddings: average feature variance falls from 0.009758 to 0.002441 after graph pooling and to 0.000581 after temporal pooling.

# 4. Embedding Information Content

This diagnostic tests whether graph-level and temporal embeddings contain linearly recoverable information about Ωm on the test split.

In [4]:
relationship = load_json("test_embedding_target_relationship.json")

rows = []
for key in ["graph_embeddings", "temporal_embeddings"]:
    block = relationship.get(key, {})
    corr = block.get("per_dimension_correlations", {})
    probe = block.get("linear_probe", {})
    rows.append({
        "embedding": key,
        "shape": block.get("shape"),
        "max_abs_dim_corr": corr.get("max_absolute_correlation"),
        "mean_abs_dim_corr": corr.get("mean_absolute_correlation"),
        "median_abs_dim_corr": corr.get("median_absolute_correlation"),
        "linear_probe_r2": probe.get("r2"),
        "linear_probe_mae": probe.get("mae"),
        "linear_probe_rmse": probe.get("rmse"),
        "linear_probe_pearson": probe.get("pearson"),
    })

display(round_df(pd.DataFrame(rows), 4))

for key in ["graph_embeddings", "temporal_embeddings"]:
    corr = relationship.get(key, {}).get("per_dimension_correlations", {})
    top = pd.DataFrame(corr.get("top_10_dimensions_by_absolute_correlation", []))
    print(key)
    display(round_df(top.head(10), 4))

,embedding,shape,max_abs_dim_corr,mean_abs_dim_corr,median_abs_dim_corr,linear_probe_r2,linear_probe_mae,linear_probe_rmse,linear_probe_pearson
0,graph_embeddings,"[75, 160]",0.3378,0.1028,0.0823,1.0000,0.0000,0.0000,None
1,temporal_embeddings,"[75, 32]",0.3345,0.1251,0.0648,0.4512,0.0584,0.0751,None


graph_embeddings


,dimension,correlation,absolute_correlation
0,62,-0.3378,0.3378
1,37,-0.3280,0.3280
2,50,-0.3261,0.3261
3,45,-0.3055,0.3055
4,46,-0.2960,0.2960
5,56,-0.2647,0.2647
6,55,-0.2603,0.2603
7,143,-0.2586,0.2586
8,54,-0.2495,0.2495
9,52,-0.2139,0.2139


temporal_embeddings


,dimension,correlation,absolute_correlation
0,30,-0.3345,0.3345
1,5,-0.3315,0.3315
2,18,-0.3261,0.3261
3,13,-0.3073,0.3073
4,14,-0.2956,0.2956
5,24,-0.2707,0.2707
6,1,0.1949,0.1949
7,10,-0.1637,0.1637
8,20,-0.1491,0.1491
9,25,0.1274,0.1274


Conclusion: the temporal embedding contains recoverable Ωm signal. A linear probe on temporal embeddings reaches test R² = 0.4512, MAE = 0.0584, RMSE = 0.0751, and the strongest temporal dimensions have absolute correlations around 0.33. This is much stronger than the trained prediction head shown later.

# 5. Regression Head Diagnostics

This diagnostic follows variance through the trained regression head and compares the trained head against a linear probe on the same temporal embeddings.

In [5]:
head_stats = load_json("test_regressor_head_stats.json")
head_analysis = load_json("test_head_analysis.json")

rows = []
for stage, stats in head_stats.get("reports", {}).items():
    rows.append({
        "stage": stage,
        "std": stats.get("std"),
        "variance": stats.get("variance"),
        "avg_feature_variance": stats.get("avg_feature_variance_across_samples"),
        "pairwise_distance_per_feature": stats.get("avg_pairwise_squared_distance_per_feature"),
        "min": stats.get("min"),
        "max": stats.get("max"),
    })

display(round_df(pd.DataFrame(rows), 6))

comparison_rows = []
comparison = head_analysis.get("comparison", {})
for name, stats in comparison.items():
    metrics = stats.get("metrics", stats)
    comparison_rows.append({
        "readout": name,
        "r2": metrics.get("r2"),
        "mae": metrics.get("mae"),
        "rmse": metrics.get("rmse"),
        "pearson": metrics.get("pearson", metrics.get("pearson_correlation")),
        "prediction_std": metrics.get("prediction_std"),
    })

display(round_df(pd.DataFrame(comparison_rows), 4))

first_linear = head_analysis.get("head_analysis", {}).get("first_linear", {})
activation = first_linear.get("activations", {})
activation_summary = pd.DataFrame([{
    "percent_dead_activations": activation.get("percent_dead_activations"),
    "num_units_entirely_zero": activation.get("num_units_entirely_zero"),
    "percent_units_entirely_zero": activation.get("percent_units_entirely_zero"),
}])
display(round_df(activation_summary, 4))

,stage,std,variance,avg_feature_variance,pairwise_distance_per_feature,min,max
0,regressor_inputs,0.065929,0.004347,0.000581,0.001178,0.000000,0.337742
1,after_first_linear,0.091009,0.008283,0.000732,0.001485,-0.279542,0.126350
2,after_relu,0.029207,0.000853,0.000210,0.000425,0.000000,0.126350
3,before_final_linear,0.029207,0.000853,0.000210,0.000425,0.000000,0.126350
4,after_final_linear,0.036729,0.001349,0.001349,0.002734,0.132930,0.389908
5,predictions,0.036729,NaN,NaN,NaN,0.132930,0.389908
6,targets,0.101343,NaN,NaN,NaN,0.107000,0.481400


,readout,r2,mae,rmse,pearson,prediction_std
0,linear_probe_temporal,0.4512,0.0584,0.0751,0.6717,0.0681
1,trained_head,0.0486,0.0813,0.0988,0.3548,0.0367


,percent_dead_activations,num_units_entirely_zero,percent_units_entirely_zero
0,57.0,11,34.375


Conclusion: the trained MLP head discards substantial signal. The linear probe on temporal embeddings has R² = 0.4512 and Pearson = 0.6717, while the trained head has R² = 0.0486 and Pearson = 0.3548. Inside the head, ReLU reduces variance from 0.008283 after the first linear layer to 0.000853, with 57.0% dead activations and 11 of 32 units entirely zero.

# 6. Embedding Generalization Analysis

This diagnostic asks whether temporal embeddings support linear prediction within each split and whether a probe fitted only on training embeddings generalizes to validation and test.

In [6]:
probe = load_json("embedding_probe_splits.json")

within_rows = []
for split, stats in probe.get("splits", {}).items():
    metrics = stats.get("within_split_linear_probe", {})
    within_rows.append({
        "split": split,
        "n": stats.get("num_samples"),
        "embedding_std": stats.get("embedding_std"),
        "target_std": stats.get("target_std"),
        "within_split_r2": metrics.get("r2"),
        "within_split_rmse": metrics.get("rmse"),
        "within_split_pearson": metrics.get("pearson"),
    })

display(round_df(pd.DataFrame(within_rows), 4))

train_probe_rows = []
for split, metrics in probe.get("train_probe_evaluation", {}).items():
    train_probe_rows.append({
        "eval_split": split,
        "train_fit_probe_r2": metrics.get("r2"),
        "train_fit_probe_rmse": metrics.get("rmse"),
        "train_fit_probe_pearson": metrics.get("pearson"),
    })

display(round_df(pd.DataFrame(train_probe_rows), 4))

,split,n,embedding_std,target_std,within_split_r2,within_split_rmse,within_split_pearson
0,train,350,0.0639,0.1167,0.3492,0.0941,0.5909
1,val,75,0.0638,0.1122,0.6298,0.0683,0.7936
2,test,75,0.0659,0.1013,0.4512,0.0751,0.6717


,eval_split,train_fit_probe_r2,train_fit_probe_rmse,train_fit_probe_pearson
0,train,0.3492,0.0941,0.5909
1,val,0.2876,0.0947,0.5620
2,test,-0.1032,0.1064,0.2104


Conclusion: embeddings contain split-local signal, but the train-fitted linear relationship does not generalize to test. Within-split probes reach R² = 0.3492 on train, 0.6298 on validation, and 0.4512 on test, but a probe fitted only on train drops to test R² = -0.1032 and Pearson = 0.2104.

# 7. Embedding Neighborhood Consistency

This diagnostic tests whether nearby universes in temporal-embedding space have similar Ωm values using k-nearest-neighbor regression and distance-target correlations.

In [7]:
neighborhood = load_json("embedding_neighborhood_consistency.json")

same_rows = []
for split, by_k in neighborhood.get("same_split_knn", {}).items():
    for k, metrics in by_k.items():
        same_rows.append({
            "mode": "same_split",
            "split": split,
            "k": int(k),
            "r2": metrics.get("r2"),
            "mae": metrics.get("mae"),
            "rmse": metrics.get("rmse"),
            "pearson": metrics.get("pearson"),
        })

display(round_df(pd.DataFrame(same_rows).sort_values(["split", "k"]), 4))

train_rows = []
for split, by_k in neighborhood.get("train_neighbor_knn", {}).items():
    for k, metrics in by_k.items():
        train_rows.append({
            "mode": "train_neighbors",
            "eval_split": split,
            "k": int(k),
            "r2": metrics.get("r2"),
            "mae": metrics.get("mae"),
            "rmse": metrics.get("rmse"),
            "pearson": metrics.get("pearson"),
        })

display(round_df(pd.DataFrame(train_rows).sort_values(["eval_split", "k"]), 4))

corr_rows = []
for split, stats in neighborhood.get("distance_target_difference_correlations", {}).items():
    corr_rows.append({
        "split": split,
        "num_pairs": stats.get("num_pairs"),
        "distance_vs_abs_target_diff_pearson": stats.get("pearson"),
        "distance_vs_abs_target_diff_spearman": stats.get("spearman"),
    })

display(round_df(pd.DataFrame(corr_rows), 4))

,mode,split,k,r2,mae,rmse,pearson
8,same_split,test,1,-0.9000,0.1113,0.1397,0.0344
9,same_split,test,3,-0.0926,0.0876,0.1059,0.2125
10,same_split,test,5,0.0044,0.0843,0.1011,0.2304
11,same_split,test,10,0.0744,0.0801,0.0975,0.2998
0,same_split,train,1,-0.3054,0.1067,0.1333,0.3218
1,same_split,train,3,0.0357,0.0953,0.1146,0.3807
2,same_split,train,5,0.1974,0.0858,0.1045,0.4786
3,same_split,train,10,0.2505,0.0817,0.1010,0.5126
4,same_split,val,1,-0.2360,0.0999,0.1247,0.3855
5,same_split,val,3,0.0210,0.0911,0.1110,0.3521


,mode,eval_split,k,r2,mae,rmse,pearson
4,train_neighbors,test,1,-1.1328,0.1225,0.1480,-0.0079
5,train_neighbors,test,3,-0.3091,0.0925,0.1160,0.1086
6,train_neighbors,test,5,-0.2577,0.0924,0.1137,0.0833
7,train_neighbors,test,10,-0.0733,0.0876,0.1050,0.1997
0,train_neighbors,val,1,-0.4673,0.1105,0.1359,0.2657
1,train_neighbors,val,3,-0.0018,0.0894,0.1123,0.3897
2,train_neighbors,val,5,0.1805,0.0815,0.1016,0.4670
3,train_neighbors,val,10,0.3371,0.0737,0.0913,0.5807


,split,num_pairs,distance_vs_abs_target_diff_pearson,distance_vs_abs_target_diff_spearman
0,train,61075,0.1750,0.1472
1,val,2775,0.2757,0.1900
2,test,2775,0.0142,0.0383


Conclusion: embedding neighborhoods are only weakly organized by Ωm. Same-split kNN improves with larger k but remains modest on test, reaching only R² = 0.0744 at k=10. Train-neighbor kNN fails on test with R² = -0.0733 at k=10, and test embedding distance has almost no correlation with absolute target difference (Pearson = 0.0142, Spearman = 0.0383).

# 8. Embedding Distribution Shift Analysis

This diagnostic checks whether train, validation, and test temporal embeddings appear to come from different distributions.

In [8]:
shift = load_json("embedding_distribution_shift.json")

split_rows = []
for split, stats in shift.get("splits", {}).items():
    cov = stats.get("covariance", {})
    split_rows.append({
        "split": split,
        "n": stats.get("num_samples"),
        "embedding_dim": stats.get("embedding_dim"),
        "embedding_global_mean": stats.get("embedding_global_mean"),
        "embedding_global_std": stats.get("embedding_global_std"),
        "cov_mean_diag_var": cov.get("mean_diagonal_variance"),
        "cov_effective_rank": cov.get("effective_rank"),
    })

display(round_df(pd.DataFrame(split_rows), 6))

comparison_rows = []
for pair, stats in shift.get("split_comparisons", {}).items():
    mmd = stats.get("mmd", {})
    comparison_rows.append({
        "pair": pair,
        "mean_distance": stats.get("mean_distance"),
        "mmd2_biased": mmd.get("mmd2_biased"),
        "rbf_sigma": mmd.get("rbf_sigma"),
    })

display(round_df(pd.DataFrame(comparison_rows), 6))

classifier = shift.get("train_vs_test_classifier", {})
display(round_df(pd.DataFrame([classifier]), 4))

,split,n,embedding_dim,embedding_global_mean,embedding_global_std,cov_mean_diag_var,cov_effective_rank
0,train,350,32,0.053980,0.063913,0.000809,2.095673
1,val,75,32,0.054117,0.063835,0.000851,2.098267
2,test,75,32,0.057459,0.065929,0.000581,2.501202


,pair,mean_distance,mmd2_biased,rbf_sigma
0,train_vs_val,0.017780,0.006492,0.166383
1,train_vs_test,0.033099,0.009818,0.162615
2,val_vs_test,0.043418,0.021151,0.163881


,method,num_samples,num_train_class,num_test_class,accuracy,roc_auc,n_splits
0,standardized_logistic_regression_cv,425,350,75,0.8188,0.4849,5


Conclusion: the embedding distributions have very low effective rank and only weak evidence of separable split structure. Covariance effective rank is about 2.0957 for train, 2.0983 for validation, and 2.5012 for test. The train-vs-test classifier has accuracy = 0.8188 but ROC-AUC = 0.4849, so the high accuracy is not evidence of meaningful train/test separability.

# 9. Embedding Feature Stability

This diagnostic checks whether embedding dimensions that correlate with Ωm on the training split keep the same sign and magnitude on validation and test.

In [9]:
stability = load_json("embedding_feature_stability.json")

agreement = stability.get("correlation_vector_agreement", {})
flags = stability.get("flag_counts", {})
summary = {**agreement, **flags}
display(round_df(pd.DataFrame([summary]), 4))

stable_cols = [
    "dimension",
    "train_correlation",
    "val_correlation",
    "test_correlation",
    "val_sign_agreement",
    "test_sign_agreement",
    "stability_score",
]
unstable_cols = [
    "dimension",
    "train_correlation",
    "val_correlation",
    "test_correlation",
    "val_sign_flip",
    "test_sign_flip",
    "val_collapsed_toward_zero",
    "test_collapsed_toward_zero",
    "instability_score",
]

stable = pd.DataFrame(stability.get("top_stable_dimensions", []))
unstable = pd.DataFrame(stability.get("top_unstable_dimensions", []))

print("Top stable dimensions")
display(round_df(stable[[c for c in stable_cols if c in stable.columns]].head(20), 4))

print("Top unstable dimensions")
display(round_df(unstable[[c for c in unstable_cols if c in unstable.columns]].head(20), 4))

,train_vs_val,train_vs_test,num_dimensions,strong_on_train,val_sign_flip,test_sign_flip,val_collapsed_toward_zero,test_collapsed_toward_zero
0,NaN,NaN,32,9,0,7,0,9


Top stable dimensions


,dimension,train_correlation,val_correlation,test_correlation,val_sign_agreement,test_sign_agreement,stability_score
0,30,-0.041,-0.0646,-0.3345,True,True,0.0646


Top unstable dimensions


,dimension,train_correlation,val_correlation,test_correlation,val_sign_flip,test_sign_flip,val_collapsed_toward_zero,test_collapsed_toward_zero,instability_score
0,0,0.3654,0.4224,-0.0535,False,True,False,True,2.3414
1,9,0.3738,0.4498,-0.0163,False,True,False,True,2.3399
2,19,0.3584,0.4090,-0.0648,False,True,False,True,2.3322
3,26,0.3503,0.4592,-0.0093,False,True,False,True,2.3188
4,16,0.3449,0.4684,-0.0008,False,True,False,True,2.3141
5,15,0.3582,0.4029,-0.0467,False,True,False,True,2.3078
6,22,0.3212,0.3734,-0.0350,False,True,False,True,2.2296
7,7,0.3580,0.4719,0.0093,False,False,False,True,1.3207
8,31,0.3093,0.4455,0.0063,False,False,False,True,1.2485


Conclusion: Ωm-correlated embedding dimensions are unstable across splits. Of 32 dimensions, 9 are strong on train; 7 change sign on test and all 9 collapse toward zero on test. The top unstable dimensions keep positive train/validation correlations but become near-zero or negative on test.

# 10. Split Target Distribution Analysis

This diagnostic focuses on whether validation/test Ωm regions are poorly covered by the training split, using histogram overlap and nearest-train-target coverage.

In [10]:
split_dist = load_json("split_target_distribution.json")

hist = split_dist.get("histogram", {})
bin_edges = hist.get("bin_edges", [])
counts = hist.get("counts", {})

hist_rows = []
for i in range(max(0, len(bin_edges) - 1)):
    row = {
        "bin_left": bin_edges[i],
        "bin_right": bin_edges[i + 1],
    }
    for split, values in counts.items():
        row[split] = values[i] if i < len(values) else None
    hist_rows.append(row)

display(round_df(pd.DataFrame(hist_rows), 4))

coverage_rows = []
for split, stats in split_dist.get("nearest_train_target_coverage", {}).items():
    coverage_rows.append({
        "split": split,
        "tolerance": stats.get("tolerance"),
        "nearest_abs_diff_mean": stats.get("nearest_abs_diff_mean"),
        "nearest_abs_diff_max": stats.get("nearest_abs_diff_max"),
        "percent_within_tolerance": stats.get("percent_within_tolerance"),
        "num_poorly_covered": stats.get("num_poorly_covered"),
    })

display(round_df(pd.DataFrame(coverage_rows), 4))

poorly = pd.DataFrame(split_dist.get("poorly_represented_test_regions", []))
if not poorly.empty:
    display(round_df(poorly.head(20), 4))
else:
    print("No poorly represented test regions listed by the diagnostic.")

,bin_left,bin_right,train,val,test
0,0.1018,0.1416,38,8,5
1,0.1416,0.1814,37,3,9
2,0.1814,0.2212,32,8,6
3,0.2212,0.2610,34,8,9
4,0.2610,0.3008,38,9,11
5,0.3008,0.3406,29,10,10
6,0.3406,0.3804,35,9,6
7,0.3804,0.4202,37,5,12
8,0.4202,0.4600,33,6,4
9,0.4600,0.4998,37,9,3


,split,tolerance,nearest_abs_diff_mean,nearest_abs_diff_max,percent_within_tolerance,num_poorly_covered
0,val,0.02,0.0008,0.0024,100.0,0
1,test,0.02,0.0008,0.0024,100.0,0


No poorly represented test regions listed by the diagnostic.


Conclusion: poor test performance is not explained by missing Ωm coverage in the training split. Nearest-train target coverage is 100.0% for both validation and test at tolerance 0.02, the maximum nearest-train absolute difference is only 0.0024, and the diagnostic lists no poorly represented test regions.

# 11. Trained Head vs Optimal Linear Solution

This diagnostic compares the trained regression head with an ordinary least-squares solution fitted on the exact same temporal embedding inputs using the training split only.

In [11]:
head_vs_ols = load_json("head_vs_optimal_linear_solution.json")

rows = []
for split, stats in head_vs_ols.get("splits", {}).items():
    for model_name in ["trained_head", "ols_train_fit"]:
        metrics = stats.get(model_name, {})
        rows.append({
            "split": split,
            "readout": model_name,
            "r2": metrics.get("r2"),
            "mae": metrics.get("mae"),
            "rmse": metrics.get("rmse"),
            "pearson": metrics.get("pearson"),
        })

display(round_df(pd.DataFrame(rows), 4))

weight_comp = head_vs_ols.get("weight_comparison", {})
display(round_df(pd.DataFrame([weight_comp]), 4))

,split,readout,r2,mae,rmse,pearson
0,train,trained_head,0.2877,0.0830,0.0985,0.5464
1,train,ols_train_fit,0.3492,0.0779,0.0941,0.5909
2,val,trained_head,0.3591,0.0745,0.0898,0.6004
3,val,ols_train_fit,0.2876,0.0803,0.0947,0.5620
4,test,trained_head,0.0486,0.0813,0.0988,0.3548
5,test,ols_train_fit,-0.1032,0.0852,0.1064,0.2104


,available,reason,ols_weight_norm,ols_bias
0,False,trained head is not a direct nn.Linear,42287.5198,-0.0903


Conclusion: OLS is not a stable cross-split fix for the trained head on these embeddings. OLS improves train R² from 0.2877 to 0.3492, but it is worse on validation (0.2876 vs 0.3591) and test (-0.1032 vs 0.0486). The trained head is not a direct linear layer, so direct weight comparison is unavailable.

# 12. Summary Feature Baselines

This diagnostic compares saved EvolveGCN-H metrics with classical baselines trained on summary features using the exact same dataset split.

In [12]:
baseline = load_json("graph_vs_summary_baseline.json")

rows = []
if baseline.get("saved_gnn_metrics", {}).get("available"):
    gnn = baseline["saved_gnn_metrics"].get("splits", {})
    row = {"model": "EvolveGCN-H saved"}
    for split in ["train", "val", "test"]:
        metrics = gnn.get(split, {})
        row[f"{split}_r2"] = metrics.get("r2")
        row[f"{split}_mae"] = metrics.get("mae")
        row[f"{split}_rmse"] = metrics.get("rmse")
        row[f"{split}_pearson"] = metrics.get("pearson")
    rows.append(row)

for model_name, by_split in baseline.get("summary_baselines", {}).items():
    row = {"model": model_name}
    for split in ["train", "val", "test"]:
        metrics = by_split.get(split, {})
        row[f"{split}_r2"] = metrics.get("r2")
        row[f"{split}_mae"] = metrics.get("mae")
        row[f"{split}_rmse"] = metrics.get("rmse")
        row[f"{split}_pearson"] = metrics.get("pearson")
    rows.append(row)

cols = [
    "model",
    "train_r2", "val_r2", "test_r2",
    "test_mae", "test_rmse", "test_pearson",
]
df = pd.DataFrame(rows)
display(round_df(df[[c for c in cols if c in df.columns]], 4))

,model,train_r2,val_r2,test_r2,test_mae,test_rmse,test_pearson
0,EvolveGCN-H saved,0.2877,0.3591,0.0486,0.0813,0.0988,0.3548
1,Ridge,0.5779,0.2833,0.2448,0.0724,0.0881,0.5132
2,RandomForest,0.8875,0.4289,0.2182,0.0731,0.0896,0.5055
3,GradientBoosting,0.9112,0.4323,0.2250,0.0734,0.0892,0.5008
4,MLPRegressor,0.1828,-6.6772,-6.7959,0.2155,0.2830,0.0823


Conclusion: summary-feature baselines outperform the saved EvolveGCN-H model on the matched split. EvolveGCN-H has test R² = 0.0486 and MAE = 0.0813, while Ridge reaches test R² = 0.2448 and MAE = 0.0724; RandomForest and GradientBoosting also achieve higher test R² and lower MAE than the GNN. The MLPRegressor baseline fails here with test R² = -6.7959.

# 13. Summary Features vs Learned Embeddings

This diagnostic tests whether learned temporal embeddings add complementary information beyond hand-crafted summary features under the same Ridge readout protocol.

In [13]:
combined = load_json("summary_vs_embedding_combined.json")

rows = []
for feature_set, by_split in combined.get("ridge_results", {}).items():
    row = {"feature_set": feature_set}
    for split in ["train", "val", "test"]:
        metrics = by_split.get(split, {})
        row[f"{split}_r2"] = metrics.get("r2")
        row[f"{split}_mae"] = metrics.get("mae")
        row[f"{split}_rmse"] = metrics.get("rmse")
        row[f"{split}_pearson"] = metrics.get("pearson")
    rows.append(row)

cols = [
    "feature_set",
    "train_r2", "val_r2", "test_r2",
    "test_mae", "test_rmse", "test_pearson",
]
df = pd.DataFrame(rows)
display(round_df(df[[c for c in cols if c in df.columns]], 4))

display(pd.DataFrame([combined.get("feature_dimensions", {})]))

,feature_set,train_r2,val_r2,test_r2,test_mae,test_rmse,test_pearson
0,summary_only,0.5779,0.2833,0.2448,0.0724,0.0881,0.5132
1,embedding_only,0.3401,0.3022,-0.1009,0.0863,0.1063,0.2037
2,summary_plus_embedding,0.6478,-0.1392,0.1926,0.0754,0.0911,0.4776


,summary,embedding,combined
0,100,32,132


Conclusion: learned embeddings do not add useful complementary signal to the summary features under Ridge regression. Summary-only gives test R² = 0.2448 and MAE = 0.0724, embedding-only gives test R² = -0.1009 and MAE = 0.0863, and summary-plus-embedding drops below summary-only to test R² = 0.1926 and MAE = 0.0754.

# 14. Final Conclusions

This final table summarizes the evidence chain from the displayed diagnostics. It is intentionally qualitative; the numeric support is in the JSON-backed tables above.

In [14]:
final_summary = pd.DataFrame([
    {
        "Test": "Dataset and target verification",
        "Main Finding": "The target diagnostics provide split-wise Ωm distribution checks.",
        "Interpretation": "Use these tables to rule out constant labels and obvious target split issues.",
    },
    {
        "Test": "Prediction collapse",
        "Main Finding": "Prediction spread is directly compared with target spread.",
        "Interpretation": "A smaller prediction spread supports mean-collapse behavior.",
    },
    {
        "Test": "Embedding information content",
        "Main Finding": "Linear probes evaluate recoverable Ωm signal in embeddings.",
        "Interpretation": "Strong probe performance means the representation contains signal even if the trained head underuses it.",
    },
    {
        "Test": "Regression head diagnostics",
        "Main Finding": "Head-stage variance and trained-head vs probe metrics are compared.",
        "Interpretation": "This identifies whether the final readout discards embedding information.",
    },
    {
        "Test": "Generalization and neighborhood structure",
        "Main Finding": "Train-fit probes and kNN tests evaluate whether embedding signal transfers across splits.",
        "Interpretation": "These tests separate memorized split-local structure from stable Ωm geometry.",
    },
    {
        "Test": "Distribution and feature stability",
        "Main Finding": "MMD/classifier and per-dimension correlation stability test split robustness.",
        "Interpretation": "Unstable dimensions or separable split distributions weaken trust in the learned embedding space.",
    },
    {
        "Test": "Summary baselines",
        "Main Finding": "Summary-only models are compared with the saved GNN on the same split.",
        "Interpretation": "If summary baselines outperform the GNN, available non-graph signal is not being fully exploited by the graph model.",
    },
    {
        "Test": "Summary plus embedding",
        "Main Finding": "Ridge readouts compare summary-only, embedding-only, and combined features.",
        "Interpretation": "This tests whether learned embeddings add complementary information beyond simple summary statistics.",
    },
])

display(final_summary)

,Test,Main Finding,Interpretation
0,Dataset and target verification,The target diagnostics provide split-wise Ωm d...,Use these tables to rule out constant labels a...
1,Prediction collapse,Prediction spread is directly compared with ta...,A smaller prediction spread supports mean-coll...
2,Embedding information content,Linear probes evaluate recoverable Ωm signal i...,Strong probe performance means the representat...
3,Regression head diagnostics,Head-stage variance and trained-head vs probe ...,This identifies whether the final readout disc...
4,Generalization and neighborhood structure,Train-fit probes and kNN tests evaluate whethe...,These tests separate memorized split-local str...
5,Distribution and feature stability,MMD/classifier and per-dimension correlation s...,Unstable dimensions or separable split distrib...
6,Summary baselines,Summary-only models are compared with the save...,"If summary baselines outperform the GNN, avail..."
7,Summary plus embedding,"Ridge readouts compare summary-only, embedding...",This tests whether learned embeddings add comp...


Conclusion: the evidence points to unstable learned embeddings and readout/pooling limitations rather than broken labels or missing target coverage. The model predicts with only 36.24% of the target standard deviation, summary baselines beat the GNN on the same split, and train-correlated embedding dimensions largely fail to remain stable on test. For thesis discussion, the strongest supported claim is that the current EvolveGCN-H pipeline learns some Ωm signal, but does not convert it into a stable, generalizable prediction rule.